# Data Preprocessing Pipeline
**ContextFlow AI** - Enterprise Knowledge Assistant

Notebook ini melakukan:
1. Load dataset dari HuggingFace
2. Text cleaning & normalization
3. Format ke instruction style
4. Deduplicate & filter
5. Train/Val split
6. Simpan ke disk

In [16]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
from datasets import load_dataset, concatenate_datasets, Dataset
from app.utils.logger import logger
from app.utils.config import config
from app.preprocessing.cleaner import clean_text, remove_duplicates, remove_missing, filter_by_length, filter_by_length_per_source
from app.preprocessing.formatter import apply_formatter, INSTRUCTION_TEMPLATE

print('All imports OK')

All imports OK


In [17]:
import shutil, os

cache_path = os.path.expanduser("~/.cache/huggingface/datasets")
for folder in os.listdir(cache_path):
    if "open_assistant" in folder.lower() or "oasst" in folder.lower():
        shutil.rmtree(os.path.join(cache_path, folder))
        print(f"Deleted: {folder}")

Deleted: OpenAssistant___oasst1


## 1. Load Datasets dari HuggingFace

In [18]:
# Import semua loader dari module yang sudah ada
from app.preprocessing.data_loader import (
    load_dolly_15k,      # loader Dolly 15K
    load_alpaca,         # loader Alpaca
    load_openassistant,  # loader OpenAssistant
    load_coqa,           # loader CoQA — sudah include flatten per Q&A pair
)

# Load semua dataset via fungsi loader masing-masing
# Jangan pakai load_dataset() langsung karena CoQA butuh flatten dulu
dolly  = load_dolly_15k()     # kolom: instruction, context, response
alpaca = load_alpaca()         # kolom: instruction, input, output
oasst  = load_openassistant()  # kolom: message_id, text, role, dll
coqa   = load_coqa()           # kolom: question, context, answer (sudah di-flatten)

# Batasi jumlah sample per dataset
SAMPLE_SIZE = 10000

raw_datasets = {
    "dolly":         dolly.select(range(min(SAMPLE_SIZE, len(dolly)))),
    "alpaca":        alpaca.select(range(min(SAMPLE_SIZE, len(alpaca)))),
    "openassistant": oasst.select(range(min(SAMPLE_SIZE, len(oasst)))),
    "coqa":          coqa.select(range(min(SAMPLE_SIZE, len(coqa)))),  # pakai coqa, bukan coqa_ds
}

# Verifikasi kolom tiap dataset sudah benar sebelum lanjut ke formatter
for name, ds in raw_datasets.items():
    print(f"{name}: {len(ds)} samples | columns: {ds.column_names}")

2026-05-20 08:38:05 | INFO | app.preprocessing.data_loader:8 - Loading Dolly 15K dataset...
2026-05-20 08:38:08 | INFO | app.preprocessing.data_loader:10 - Dolly 15K loaded: 15011 samples
2026-05-20 08:38:08 | INFO | app.preprocessing.data_loader:15 - Loading Alpaca dataset...
2026-05-20 08:38:13 | INFO | app.preprocessing.data_loader:17 - Alpaca loaded: 52002 samples
2026-05-20 08:38:13 | INFO | app.preprocessing.data_loader:35 - Loading OpenAssistant dataset...


Generating validation split: 100%|██████████| 4401/4401 [00:00<00:00, 319527.99 examples/s]

2026-05-20 08:38:19 | INFO | app.preprocessing.data_loader:37 - OpenAssistant raw loaded: 84437 messages


2026-05-20 08:38:47 | INFO | app.preprocessing.data_loader:75 - OpenAssistant loaded & paired: 52912 Q&A pairs
2026-05-20 08:38:47 | INFO | app.preprocessing.data_loader:86 - Loading CoQA dataset...
2026-05-20 08:38:55 | INFO | app.preprocessing.data_loader:104 - CoQA loaded & flattened: 108647 samples
dolly: 10000 samples | columns: ['instruction', 'context', 'response', 'category']
alpaca: 10000 samples | columns: ['instruction', 'input', 'output', 'text']
openassistant: 10000 samples | columns: ['instruction', 'response']
coqa: 10000 samples | columns: ['question', 'context', 'answer']


## 2. Sampling & Formatting

In [19]:
SAMPLE_SIZE = 10000

raw_datasets = {
    'dolly': dolly.select(range(min(SAMPLE_SIZE, len(dolly)))),
    'alpaca': alpaca.select(range(min(SAMPLE_SIZE, len(alpaca)))),
    'openassistant': oasst.select(range(min(SAMPLE_SIZE, len(oasst)))),
    'coqa': coqa.select(range(min(SAMPLE_SIZE, len(coqa)))),
}

for name, ds in raw_datasets.items():
    print(f'{name}: {len(ds)} samples')

dolly: 10000 samples
alpaca: 10000 samples
openassistant: 10000 samples
coqa: 10000 samples


In [20]:
print(raw_datasets["coqa"][0])
print(raw_datasets["coqa"].column_names)

{'question': 'When was the Vat formally opened?', 'context': 'The Vatican Apostolic Library (), more commonly called the Vatican Library or simply the Vat, is the library of the Holy See, located in Vatican City. Formally established in 1475, although it is much older, it is one of the oldest libraries in the world and contains one of the most significant collections of historical texts. It has 75,000 codices from throughout history, as well as 1.1 million printed books, which include some 8,500 incunabula. \n\nThe Vatican Library is a research library for history, law, philosophy, science and theology. The Vatican Library is open to anyone who can document their qualifications and research needs. Photocopies for private study of pages from books published between 1801 and 1990 can be requested in person or by mail. \n\nIn March 2014, the Vatican Library began an initial four-year project of digitising its collection of manuscripts, to be made available online. \n\nThe Vatican Secret A

In [21]:
# Format semua dataset ke instruction style
formatted = []
for source, ds in raw_datasets.items():
    try:
        fmt = apply_formatter(ds, source)
        formatted.append(fmt)
        print(f'{source}: {len(fmt)} formatted')
    except Exception as e:
        print(f'Error formatting {source}: {e}')

combined = concatenate_datasets(formatted)
print(f'\nTotal combined: {len(combined)} samples')

2026-05-20 08:38:55 | INFO | app.preprocessing.formatter:158 - Formatted 10000 samples from dolly
dolly: 10000 formatted
2026-05-20 08:38:55 | INFO | app.preprocessing.formatter:158 - Formatted 10000 samples from alpaca
alpaca: 10000 formatted


Map: 100%|██████████| 10000/10000 [00:00<00:00, 12909.27 examples/s]

2026-05-20 08:38:56 | INFO | app.preprocessing.formatter:158 - Formatted 10000 samples from openassistant


openassistant: 10000 formatted


Map: 100%|██████████| 10000/10000 [00:00<00:00, 14483.63 examples/s]

2026-05-20 08:38:56 | INFO | app.preprocessing.formatter:158 - Formatted 10000 samples from coqa
coqa: 10000 formatted

Total combined: 40000 samples


In [22]:
# Lihat contoh hasil format
print(combined[0]['text'])

### Instruction:
When did Virgin Australia start operating?

### Input:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.

### Response:
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.


In [23]:
# Cek distribusi output kosong per source
df = combined.to_pandas()
print(df.groupby('source')['output'].apply(lambda x: (x.str.strip() == "").sum()))

source
alpaca           7
coqa             0
dolly            0
openassistant    0
Name: output, dtype: int64


In [24]:
# Sebelum combined.to_pandas(), cek hasil formatter CoQA
coqa_formatted = raw_datasets["coqa"]
fmt = apply_formatter(coqa_formatted, "coqa")
print(fmt[0])                        # lihat semua kolom hasil formatter
print(repr(fmt[0].get("output")))    # lihat exact value output
print(fmt.column_names)              # pastiin kolom 'output' ada

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map: 100%|██████████| 10000/10000 [00:00<00:00, 14577.21 examples/s]

2026-05-20 08:38:57 | INFO | app.preprocessing.formatter:158 - Formatted 10000 samples from coqa
{'instruction': 'When was the Vat formally opened?', 'input': 'The Vatican Apostolic Library (), more commonly called the Vatican Library or simply the Vat, is the library of the Holy See, located in Vatican City. Formally established in 1475, although it is much older, it is one of the oldest libraries in the world and contains one of the most significant collections of historical texts. It has 75,000 codices from throughout history, as well as 1.1 million printed books, which include some 8,500 incunabula. The Vatican Library is a research library for h', 'output': 'It was formally established in 1475', 'text': '<|system|>\nYou are a helpful assistant.</s>\n<|user|>\nWhen was the Vat formally opened?\nThe Vatican Apostolic Library (), more commonly called the Vatican Library or simply the Vat, is the library of the Holy See, located in Vatican City. Formally established in 1475, although 

## 3. Text Cleaning

In [25]:
from app.preprocessing.cleaner import (
    remove_missing,           # hapus baris dengan nilai kosong
    remove_duplicates,        # hapus duplikat
    filter_by_length_per_source,  # filter panjang secara source-aware
)

# Konversi combined HuggingFace Dataset → pandas DataFrame untuk cleaning
df = combined.to_pandas()
print(f"Before cleaning: {len(df)} rows")
print(df.groupby("source").size())  # lihat distribusi awal per source

# === STEP 1: Hapus missing values ===
# Kolom 'output' untuk openassistant sengaja kosong (by design), tidak ikut difilter
# Kolom 'instruction' wajib ada untuk semua source
df = remove_missing(df, required_cols=["instruction", "output"])

# === STEP 2: Hapus duplikat ===
# Kombinasi instruction + output yang identik dianggap duplikat
df = remove_duplicates(df, subset=["instruction", "output"])

# === STEP 3: Filter panjang secara source-aware ===
# Setiap source punya threshold berbeda (lihat LENGTH_CONFIG di cleaner.py):
# - CoQA    : min=1   karena jawaban memang 1-5 kata ("research", "1475", "five")
# - Dolly   : min=10  karena response deskriptif panjang
# - Alpaca  : min=10  sama seperti dolly
# - OA      : min=0   output kosong by design, jangan difilter
# - Company : min=5   fleksibel
df = filter_by_length_per_source(df, col="output")

# === STEP 4: Filter panjang instruction (berlaku semua source) ===
# Instruction minimal 5 karakter — terlalu pendek tidak bermakna
df = df[df["instruction"].str.len() >= 5]

# Reset index setelah semua filter
df = df.reset_index(drop=True)

print(f"\nAfter cleaning: {len(df)} rows")
print(df.groupby("source").size())  # verifikasi semua source masih ada

Before cleaning: 40000 rows
source
alpaca           10000
coqa             10000
dolly            10000
openassistant    10000
dtype: int64
2026-05-20 08:38:57 | INFO | app.preprocessing.cleaner:60 - Removed 10 missing/empty rows
2026-05-20 08:38:57 | INFO | app.preprocessing.cleaner:28 - Removed 39 duplicates
2026-05-20 08:38:57 | INFO | app.preprocessing.cleaner:116 - [dolly] filter 'output': min=10, max=1500 → 9399 rows
2026-05-20 08:38:57 | INFO | app.preprocessing.cleaner:116 - [alpaca] filter 'output': min=10, max=1500 → 9423 rows
2026-05-20 08:38:57 | INFO | app.preprocessing.cleaner:116 - [openassistant] filter 'output': min=0, max=9999 → 9992 rows
2026-05-20 08:38:58 | INFO | app.preprocessing.cleaner:116 - [coqa] filter 'output': min=1, max=500 → 9978 rows
2026-05-20 08:38:58 | INFO | app.preprocessing.cleaner:139 - Total after filter_by_length_per_source: 38792 rows

After cleaning: 38527 rows
source
alpaca           9423
coqa             9758
dolly            9399
openassis

In [26]:
# Contoh data bersih
df[['instruction', 'output', 'source']].head(10)

,instruction,output,source
0,When did Virgin Australia start operating?,Virgin Australia commenced services on 31 Augu...,dolly
1,Why can camels survive for long without water?,Camels use the fat in their humps to keep them...,dolly
2,"Alice's parents have three daughters: Amy, Jes...",The name of the third daughter is Alice,dolly
3,When was Tomoaki Komorida born?,"Tomoaki Komorida was born on July 10,1981.",dolly
4,If I have more pieces at the time of stalemate...,No. Stalemate is a drawn position. It doesn't ...,dolly
5,"Given a reference text about Lollapalooza, whe...",Lollapalooze is an annual musical festival hel...,dolly
6,Who gave the UN the land in NY to build their HQ,John D Rockerfeller,dolly
7,Why mobile is bad for human,We are always engaged one phone which is not g...,dolly
8,Who was John Moses Browning?,John Moses Browning is one of the most well-kn...,dolly
9,Who was Kyle Van Zyl playing against when he s...,Kyle Van Zyl was playing against Boland U21 wh...,dolly


In [27]:
df['source'].unique()

array(['dolly', 'alpaca', 'openassistant', 'coqa'], dtype=object)

## 4. Train/Val Split

In [28]:
split_idx = int(len(df) * 0.9)
train_df = df.iloc[:split_idx]
val_df = df.iloc[split_idx:]

print(f'Train: {len(train_df)} samples')
print(f'Val: {len(val_df)} samples')

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)

Train: 34674 samples
Val: 3853 samples


---
## 6. Company Data Processing

Bagian ini memproses **semua data perusahaan** dari folder `data/raw/company/`.
Format yang didukung: CSV (business dataset), JSON, PDF, DOCX, TXT, dan gambar (OCR).

Data company akan digabungkan dengan dataset yang sudah di-preprocess di atas,
kemudian disimpan ulang sebagai train/val split yang lengkap.

In [32]:
# === Load Company Data dari folder data/raw/company/ ===
from app.preprocessing.company_loader import load_company_data

company_ds = load_company_data()

if company_ds is not None and len(company_ds) > 0:
    print(f'Company data loaded: {len(company_ds)} QA pairs')
    print(f'Columns: {company_ds.column_names}')
    print(f'Sample: {company_ds[0]}')
else:
    print('Tidak ada company data yang ditemukan di data/raw/company/')
    company_ds = None

2026-05-20 08:39:41 | INFO | app.preprocessing.company_loader:215 -   Generated 1000 Q&A pairs from business CSV amsterdam-business-dataset-sample.csv
2026-05-20 08:39:41 | INFO | app.preprocessing.company_loader:215 -   Generated 1000 Q&A pairs from business CSV berlin-business-dataset-sample.csv
2026-05-20 08:39:41 | INFO | app.preprocessing.company_loader:215 -   Generated 1000 Q&A pairs from business CSV london-business-dataset-sample.csv
2026-05-20 08:39:41 | INFO | app.preprocessing.company_loader:215 -   Generated 1000 Q&A pairs from business CSV los-angeles-business-dataset-sample.csv
2026-05-20 08:39:41 | INFO | app.preprocessing.company_loader:215 -   Generated 1000 Q&A pairs from business CSV madrid-business-dataset-sample.csv
2026-05-20 08:39:41 | INFO | app.preprocessing.company_loader:215 -   Generated 1000 Q&A pairs from business CSV new-york-city-business-dataset-sample.csv
2026-05-20 08:39:41 | INFO | app.preprocessing.company_loader:215 -   Generated 1000 Q&A pairs fr

In [33]:
# === Format Company Data ke instruction style ===
# Menggunakan formatter yang sama seperti dataset lainnya

if company_ds is not None:
    company_formatted = apply_formatter(company_ds, 'company')
    print(f'Company formatted: {len(company_formatted)} samples')
    print(f'Columns: {company_formatted.column_names}')
    print(f'\nSample text:\n{company_formatted[0]["text"][:500]}...')
else:
    company_formatted = None
    print('Skip formatting — tidak ada company data')

Map: 100%|██████████| 8005/8005 [00:00<00:00, 13288.38 examples/s]

2026-05-20 08:39:48 | INFO | app.preprocessing.formatter:158 - Formatted 8005 samples from company
Company formatted: 8005 samples
Columns: ['instruction', 'input', 'output', 'text', 'source']

Sample text:
<|system|>
You are a helpful assistant.</s>
<|user|>
Tell me about the company Cbre Gws Integrated Facility Management B.V. in Amsterdam.
Company: Cbre Gws Integrated Facility Management B.V. Address: Anthony Fokkerweg 15, 1059 CM Amsterdam, Netherlands Business Category: Offices of Holding Companies, Not Elsewhere Classified Founded: 2019 Annual Revenue (USD): $320905361 Total Employees: 112000 Legal Type: Corporation</s>
<|assistant|>
Cbre Gws Integrated Facility Management B.V. is a company i...


In [34]:
# === Gabungkan Company Data dengan Dataset Lainnya ===
# 'combined' adalah hasil dari section sebelumnya (dolly + alpaca + oasst + coqa)

if company_formatted is not None:
    combined_all = concatenate_datasets([combined, company_formatted])
    print(f'Combined (with company): {len(combined_all)} samples')
else:
    combined_all = combined
    print(f'Combined (without company): {len(combined_all)} samples')

# Lihat distribusi per source
df_all = combined_all.to_pandas()
print(f'\nDistribusi per source:')
print(df_all.groupby('source').size())

Combined (with company): 48005 samples

Distribusi per source:
source
alpaca           10000
company           8005
coqa             10000
dolly            10000
openassistant    10000
dtype: int64


In [35]:
# === Cleaning Data Gabungan (termasuk Company) ===
# Proses yang sama seperti di Section 3, tapi sekarang termasuk company data

print(f'Before cleaning: {len(df_all)} rows')

# STEP 1: Hapus missing values
df_all = remove_missing(df_all, required_cols=['instruction', 'output'])

# STEP 2: Hapus duplikat
df_all = remove_duplicates(df_all, subset=['instruction', 'output'])

# STEP 3: Filter panjang secara source-aware
df_all = filter_by_length_per_source(df_all, col='output')

# STEP 4: Filter panjang instruction (berlaku semua source)
df_all = df_all[df_all['instruction'].str.len() >= 5]

# Reset index
df_all = df_all.reset_index(drop=True)

print(f'\nAfter cleaning: {len(df_all)} rows')
print(df_all.groupby('source').size())

Before cleaning: 48005 rows
2026-05-20 08:39:56 | INFO | app.preprocessing.cleaner:60 - Removed 10 missing/empty rows
2026-05-20 08:39:56 | INFO | app.preprocessing.cleaner:28 - Removed 53 duplicates
2026-05-20 08:39:56 | INFO | app.preprocessing.cleaner:116 - [dolly] filter 'output': min=10, max=1500 → 9399 rows
2026-05-20 08:39:56 | INFO | app.preprocessing.cleaner:116 - [alpaca] filter 'output': min=10, max=1500 → 9423 rows
2026-05-20 08:39:56 | INFO | app.preprocessing.cleaner:116 - [openassistant] filter 'output': min=0, max=9999 → 9992 rows
2026-05-20 08:39:56 | INFO | app.preprocessing.cleaner:116 - [coqa] filter 'output': min=1, max=500 → 9978 rows
2026-05-20 08:39:56 | INFO | app.preprocessing.cleaner:116 - [company] filter 'output': min=5, max=1500 → 7991 rows
2026-05-20 08:39:56 | INFO | app.preprocessing.cleaner:139 - Total after filter_by_length_per_source: 46783 rows

After cleaning: 46518 rows
source
alpaca           9423
company          7991
coqa             9758
dolly

In [36]:
# === Verifikasi Data Company Setelah Cleaning ===
company_rows = df_all[df_all['source'] == 'company']
print(f'Company samples setelah cleaning: {len(company_rows)}')

if len(company_rows) > 0:
    print('\n--- Sample Company Data ---')
    for i in range(min(3, len(company_rows))):
        row = company_rows.iloc[i]
        print(f'\n[{i+1}] Instruction: {row["instruction"][:100]}...')
        print(f'    Output: {row["output"][:100]}...')
        print(f'    Source: {row["source"]}')

Company samples setelah cleaning: 7991

--- Sample Company Data ---

[1] Instruction: Tell me about the company Cbre Gws Integrated Facility Management B.V. in Amsterdam....
    Output: Cbre Gws Integrated Facility Management B.V. is a company in the Offices of Holding Companies, Not E...
    Source: company

[2] Instruction: Tell me about the company Bavella B.V. in Amsterdam....
    Output: Bavella B.V. is a company in the Offices of Holding Companies, Not Elsewhere Classified sector locat...
    Source: company

[3] Instruction: Tell me about the company Vox Ventures B.V. in Amsterdam....
    Output: Vox Ventures B.V. is a company in the Offices of Holding Companies, Not Elsewhere Classified sector ...
    Source: company


### 6.1 Re-split dan Simpan Ulang (Termasuk Company Data)

In [37]:
# === Train/Val Split (Termasuk Company Data) ===
split_idx = int(len(df_all) * 0.9)
train_df_all = df_all.iloc[:split_idx]
val_df_all = df_all.iloc[split_idx:]

print(f'Train: {len(train_df_all)} samples')
print(f'Val: {len(val_df_all)} samples')
print(f'\nTrain sources: {train_df_all["source"].value_counts().to_dict()}')
print(f'Val sources: {val_df_all["source"].value_counts().to_dict()}')

train_ds_all = Dataset.from_pandas(train_df_all, preserve_index=False)
val_ds_all = Dataset.from_pandas(val_df_all, preserve_index=False)

Train: 41866 samples
Val: 4652 samples

Train sources: {'openassistant': 9947, 'coqa': 9758, 'alpaca': 9423, 'dolly': 9399, 'company': 3339}
Val sources: {'company': 4652}


In [39]:
# === Simpan ke Disk ===
os.makedirs(config.FORMATTED_DATA_DIR, exist_ok=True)

train_ds_all.save_to_disk(os.path.join(config.FORMATTED_DATA_DIR, 'train'))
val_ds_all.save_to_disk(os.path.join(config.FORMATTED_DATA_DIR, 'val'))

print(f'Saved train ({len(train_ds_all)}) and val ({len(val_ds_all)}) to {config.FORMATTED_DATA_DIR}')

Saving the dataset (1/1 shards): 100%|██████████| 4652/4652 [00:00<00:00, 710816.11 examples/s]

Saved train (41866) and val (4652) to D:\bootcamp\final_project_fine_tuning\contextflow_ai_fine_tuning\data\formatted


In [40]:
# === Verifikasi Final ===
from datasets import load_from_disk

train_check = load_from_disk(os.path.join(config.FORMATTED_DATA_DIR, 'train'))
val_check = load_from_disk(os.path.join(config.FORMATTED_DATA_DIR, 'val'))

print(f'Loaded train: {len(train_check)} samples')
print(f'Loaded val: {len(val_check)} samples')
print(f'\nTrain sources: {pd.Series([s for s in train_check["source"]]).value_counts().to_dict()}')
print(f'Columns: {train_check.column_names}')
print(f'\nSample (first):')
print(train_check[0])

Loaded train: 41866 samples
Loaded val: 4652 samples

Train sources: {'openassistant': 9947, 'coqa': 9758, 'alpaca': 9423, 'dolly': 9399, 'company': 3339}
Columns: ['instruction', 'input', 'output', 'text', 'source']

Sample (first):
{'instruction': 'When did Virgin Australia start operating?', 'input': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'output': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'text': "### Instruction:\nWhen did Virgin Australia start op

### 6.2 Ringkasan Preprocessing Lengkap

| Dataset | Source | Format |
|---------|--------|--------|
| Dolly 15K | HuggingFace | instruction, context, response |
| Alpaca | HuggingFace | instruction, input, output |
| OpenAssistant | HuggingFace | tree-based → paired Q&A |
| CoQA | HuggingFace | story + lists → flattened Q&A |
| **Company** | **Local files** | **CSV, JSON, PDF, DOCX, TXT, Image** |

Semua dataset diproses melalui pipeline yang sama:
1. Load → Format → Clean → Split → Save
2. Output tersimpan di `data/formatted/train` dan `data/formatted/val`